In [2]:
!rm -rf /content/Untitled Folder
!rm -rf /content/braille_images/
!rm -rf /content/images
!rm -rf /content/user_results/
!rm -rf /content/user_codes

In [ ]:
from google.colab import files
from google.colab.patches import cv2_imshow
from PIL import Image, ImageOps
from pathlib import Path
import cv2
import numpy as np

uploaded = files.upload()
filename = next(iter(uploaded))

BASE_DIR = Path("/content")
CODES = "binary_order"
OUTPUT_DIR = Path("/content/user_codes")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = OUTPUT_DIR / f"{CODES}.txt"
image = cv2.imdecode(
    np.frombuffer(uploaded[filename], dtype=np.uint8),
    cv2.IMREAD_COLOR
)

if image is None:
    raise ValueError("Couldn't read the file.")

def fit_without_crop(source, target_size):
    source = ImageOps.exif_transpose(source).convert("RGBA")


    fitted = ImageOps.contain(
        source,
        target_size,
        method=Image.Resampling.LANCZOS,
    )

    result = Image.new("RGBA", target_size, "white")

    x = (target_size[0] - fitted.width) // 2
    y = (target_size[1] - fitted.height) // 2
    result.alpha_composite(fitted, dest=(x, y))

    return result.convert("RGB")


TARGET_SIZE = (440, 300)

source = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
prepared_image = fit_without_crop(source, TARGET_SIZE)

prepared_image = cv2.cvtColor(
    np.array(prepared_image),
    cv2.COLOR_RGB2BGR,
)


target_w, target_h = 44, 30
h, w = prepared_image.shape[:2]

#Resize to fill the target area completely.
scale = max(target_w / w, target_h / h)

new_w = max(target_w, round(w * scale))
new_h = max(target_h, round(h * scale))

resized = cv2.resize(
    prepared_image,
    (new_w, new_h),
    interpolation=cv2.INTER_AREA if scale < 1 else cv2.INTER_CUBIC
)

# Trim the excess parts based on the center.
x = (new_w - target_w) // 2
y = (new_h - target_h) // 2

cropped = resized[y:y + target_h, x:x + target_w]


gray = cv2.cvtColor(cropped, cv2.COLOR_BGR2GRAY)

# Convert to black and white using a brightness threshold of 50

_, binary = cv2.threshold(gray, 235, 255, cv2.THRESH_BINARY)

# Black (0) → 1, white (255) → 0
matrix = (binary == 0).astype(np.uint8)

h, w = matrix.shape

# Check that the width is divisible by 2 and the height by 3
if h % 3 != 0 or w % 2 != 0:
    raise ValueError("The width must be the multiple of 2 and the height must be the multiple of 3.")

# (Block rows, block columns, 3 rows within each block, 2 columns within each block)
sets = matrix.reshape(h // 3, 3, w // 2, 2).transpose(0, 2, 1, 3)

# Output row by row, starting from the top-left corner
with open(OUTPUT_FILE, "a", encoding="utf-8") as file:
  for row in range(sets.shape[0]):
      for col in range(sets.shape[1]):
          code = ''.join(map(str, sets[row, col].flatten()))
          file.write(code + '\n')

In [ ]:
import csv
import re

from PIL import Image, ImageDraw

CODES = "binary_order"
RESULTS = "braille_translated"
BASE_DIR = Path("/content")
LAYOUT_FILE = Path("/content/user_codes") / f"{CODES}.txt"
OUTPUT_DIR = BASE_DIR / "user_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)  # <-- Creates /content/user_results
OUTPUT_FILE = OUTPUT_DIR / f"{RESULTS}.png"
IMAGE_DIR = BASE_DIR / "braille_images"

ROWS = 10
COLS = 22


GAP_X = 10
GAP_Y = 10
MARGIN = 20

def generate_braille_tiles(output_dir, cell_size=(40, 60), dot_radius=5):
    """Generates all 64 binary combination PNGs for 6-dot Braille."""
    output_dir.mkdir(parents=True, exist_ok=True)
    w, h = cell_size

    # Standard 6-dot positions (2 columns x 3 rows)
    # [dot1, dot2, dot3, dot4, dot5, dot6]
    dot_positions = [
        (w // 3, h // 4),       # Dot 1 (top-left)
        (w // 3, h // 2),       # Dot 2 (mid-left)
        (w // 3, 3 * h // 4),   # Dot 3 (bot-left)
        (2 * w // 3, h // 4),   # Dot 4 (top-right)
        (2 * w // 3, h // 2),   # Dot 5 (mid-right)
        (2 * w // 3, 3 * h // 4)# Dot 6 (bot-right)
    ]

    for i in range(64):
        # Convert integer to 6-bit binary string (e.g., 0 -> "000000", 63 -> "111111")
        code = f"{i:06b}"
        img = Image.new("RGBA", (w, h), (255, 255, 255, 255))
        draw = ImageDraw.Draw(img)

        for bit_idx, bit in enumerate(code):
            if bit == '1':
                cx, cy = dot_positions[bit_idx]
                draw.ellipse(
                    [cx - dot_radius, cy - dot_radius, cx + dot_radius, cy + dot_radius],
                    fill=(0, 0, 0, 255)
                )

        img.save(output_dir / f"{code}.png")

# Run this prior to running main()
generate_braille_tiles(IMAGE_DIR)


def main():
    with LAYOUT_FILE.open(encoding="utf-8-sig") as file:
        codes = [
            line.strip()
            for line in file
            if line.strip()
        ]

    # Check the number of codes

    if len(codes) != ROWS * COLS:
        raise ValueError(
            f"Expected {ROWS * COLS} braille codes. "
            f"Found {len(codes)}."
        )

    # Check that each code consists of exactly 6 binary digits (0 or 1)
    for line_number, code in enumerate(codes, start=1):
        if re.fullmatch(r"[01]{6}", code) is None:
            raise ValueError(
                f"Invalid code at position {line_number}: {code!r}"
            )

    # Group the 220 codes into 10 rows of 22 columns

    layout = [
        codes[start:start + COLS]
        for start in range(0, len(codes), COLS)
    ]

    # Keep the existing image loading, compositing, and saving code below.


    # Check that the layout has 10 rows and 22 columns

    if len(layout) != ROWS:
        raise ValueError(
            f"The layout must have {ROWS} rows. "
            f"Found {len(layout)}."
        )

    for row_index, row in enumerate(layout, start=1):
        if len(row) != COLS:
            raise ValueError(
                f"Row {row_index} must contain {COLS} codes. "
                f"Found {len(row)}."
            )

        for col_index, code in enumerate(row, start=1):
            if re.fullmatch(r"[01]{6}", code) is None:
                raise ValueError(
                    f"Invalid code at row {row_index}, "
                    f"column {col_index}: {code!r}"
                )

    # Load each required image only once
    images = {}
    cell_size = None

    for code in sorted({code for row in layout for code in row}):
        path = IMAGE_DIR / f"{code}.png"

        if not path.is_file():
            raise FileNotFoundError(f"Braille image not found: {path}")

        with Image.open(path) as source:
            tile = source.convert("RGBA")

        if cell_size is None:
            cell_size = tile.size
        elif tile.size != cell_size:
            raise ValueError(
                f"{path.name} has dimensions {tile.size}. "
                f"All images must have dimensions {cell_size}."
            )

        images[code] = tile

    # Create the full canvas

    cell_width, cell_height = cell_size

    canvas_width = (
        MARGIN * 2 + COLS * cell_width + (COLS - 1) * GAP_X
    )
    canvas_height = (
        MARGIN * 2 + ROWS * cell_height + (ROWS - 1) * GAP_Y
    )

    canvas = Image.new(
        "RGBA",
        (canvas_width, canvas_height),
        (255, 255, 255, 255),
    )

    # Arrange the 220 images from top to bottom and left to right
    for row_index, row in enumerate(layout):
        for col_index, code in enumerate(row):
            x = MARGIN + col_index * (cell_width + GAP_X)
            y = MARGIN + row_index * (cell_height + GAP_Y)

            canvas.alpha_composite(images[code], dest=(x, y))

    # Save as PNG
    canvas.convert("RGB").save(OUTPUT_FILE)
    print(f"Done: {ROWS * COLS} braille cells → {OUTPUT_FILE}")

if __name__ == "__main__":
    main()

Done: 220 braille cells → /content/user_results/braille_translated.png


In [ ]:
# from PIL import Image, ImageOps

# def fit_without_crop(source, target_size):
#     source = ImageOps.exif_transpose(source).convert("RGBA")


#     fitted = ImageOps.contain(
#         source,
#         target_size,
#         method=Image.Resampling.LANCZOS,
#     )

#     result = Image.new("RGBA", target_size, "white")

#     x = (target_size[0] - fitted.width) // 2
#     y = (target_size[1] - fitted.height) // 2
#     result.alpha_composite(fitted, dest=(x, y))

#     return result.convert("RGB")

# INPUT_FILE = BASE_DIR / "/content/Screenshot 2026-09-26 at 2.10.57 PM (1).png"
# TARGET_SIZE = (440, 300)

# with Image.open(INPUT_FILE) as source:
#     prepared_image = fit_without_crop(source, TARGET_SIZE)

# prepared_image.save(Path("/content/sample_data")/ "prepared_image.png")


In [ ]:
from pathlib import Path
from PIL import Image

output_dir = Path("images")
output_dir.mkdir(parents=True, exist_ok=True)

# (x, y) coordinates for each bit — the top-left corner is (0, 0)
positions = [
    (1, 1), (3, 1),
    (1, 3), (3, 3),
    (1, 5), (3, 5),
]

for number in range(64):
    bits = f"{number:06b}"  # 000000 to 111111

    # 5 pixels wide × 7 pixels high, with a white background
    image = Image.new("RGB", (5, 7), color="white")

    for bit, position in zip(bits, positions):
        color = (0, 0, 0) if bit == "1" else (255, 255, 255)
        image.putpixel(position, color)

    output_dir = Path("/content/Untitled Folder 1")
    output_dir.mkdir(parents=True, exist_ok=True)
    image = image.resize((35, 49), resample=Image.Resampling.NEAREST)
    image.save(output_dir / f"{bits}.png")

print(f"Successfully generated 64 images: {output_dir.resolve()}")

Successfully generated 64 images: /content/Untitled Folder 1


In [ ]:
!rm -rf /content/Untitled Folder

In [ ]:
%pip install py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.7/98.7 kB 3.4 MB/s eta 0:00:00


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

print("OpenCV version:", cv2.__version__)

OpenCV version: 5.0.0


In [ ]:
from pathlib import Path

FILE_NAME = "input_letters1"
W2B = "test_result4"

INPUT_TEXT = Path("/content/user_text") / f"{FILE_NAME}.txt"

# 1. Ensure parent directory exists and write sample text if and only if missing
INPUT_TEXT.parent.mkdir(parents=True, exist_ok=True)
if not INPUT_TEXT.is_file():
    INPUT_TEXT.write_text("your text to translate here", encoding="utf-8")

# Remove any leading BOM and replace tabs with spaces
text = INPUT_TEXT.read_text(encoding="utf-8-sig").expandtabs(4)
print(text)
# English lowercase letters
letters = dict(zip(
    "abcdefghijklmnopqrstuvwxyz",
    "⠁⠃⠉⠙⠑⠋⠛⠓⠊⠚⠅⠇⠍⠝⠕⠏⠟⠗⠎⠞⠥⠧⠺⠭⠽⠵"
))

# English uppercase letters
letters.update({
    char.upper(): "⠠" + braille
    for char, braille in list(letters.items())
})

punctuation = {
    " ": " ",
    ",": "⠂",
    ".": "⠲",
    ":": "⠒",
    "\n": "\n",
    "(": "⠐⠣",
    ")": "⠐⠜",
    "[": "⠨⠣",
    "]": "⠨⠜",
    "{": "⠸⠣",
    "}": "⠸⠜",
    "′": "⠶",
    "″": "⠶⠶",
    "‴": "⠶⠶⠶",
    "=": "⠐⠶",
}

# Greek letters in UEB technical notation
greek_letters = {
    "α": "⠨⠁",  # alpha
    "β": "⠨⠃",  # beta
    "γ": "⠨⠛",  # gamma
    "δ": "⠨⠙",  # delta
    "ε": "⠨⠑",  # epsilon
    "ζ": "⠨⠵",  # zeta
    "η": "⠨⠱",  # eta
    "θ": "⠨⠹",  # theta
    "ι": "⠨⠊",  # iota
    "κ": "⠨⠅",  # kappa
    "λ": "⠨⠇",  # lambda
    "μ": "⠨⠍",  # mu
    "ν": "⠨⠝",  # nu
    "ξ": "⠨⠭",  # xi
    "ο": "⠨⠕",  # omicron
    "π": "⠨⠏",  # pi
    "ρ": "⠨⠗",  # rho
    "σ": "⠨⠎",  # sigma
    "τ": "⠨⠞",  # tau
    "υ": "⠨⠥",  # upsilon
    "φ": "⠨⠋",  # phi
    "χ": "⠨⠯",  # chi
    "ψ": "⠨⠽",  # psi
    "ω": "⠨⠺",  # omega
}

# Greek uppercase letters
greek_letters.update({
    char.upper(): "⠠" + braille
    for char, braille in list(greek_letters.items())
})

# Final sigma
greek_letters["ς"] = greek_letters["σ"]

# Digits use a numeric indicator before each number
digits = dict(zip("1234567890", "⠁⠃⠉⠙⠑⠋⠛⠓⠊⠚"))

punctuation.update({
    "+": "⠐⠖",
    "−": "⠐⠤",   # Mathematical minus (U+2212)
    "-": "⠤",     # Hyphen
    "/": "⠸⠌",   # Printed slash
    "<": "⠈⠣",
    ">": "⠈⠜",
    "'": "⠄",     # Apostrophe; use ′ for a prime
    "“": "⠦",
    "”": "⠴",
})


def to_braille(text):
    result = []
    opening_quote = True
    numeric_mode = False

    for index, char in enumerate(text):
        # Add one numeric indicator per consecutive number
        if char in digits:
            if not numeric_mode:
                result.append("⠼")
            result.append(digits[char])
            numeric_mode = True
            continue

        # A period or comma preserves numeric mode
        if numeric_mode and char in ".,":
            result.append(punctuation[char])
            continue

        # Distinguish letters a–j from digits after a number
        if numeric_mode and char in "abcdefghij":
            result.append("⠰")

        numeric_mode = False

        if char in greek_letters:
            result.append(greek_letters[char])

        elif char == '"':
            result.append("⠦" if opening_quote else "⠴")
            opening_quote = not opening_quote

        elif char in punctuation:
            result.append(punctuation[char])

        elif char in letters:
            result.append(letters[char])

        else:
            raise ValueError(
                f"Unsupported character at position {index + 1}: "
                f"{char!r} (U+{ord(char):04X})"
            )

    return "".join(result)

braille_text = to_braille(text)

output_path = (
    Path("/content/user_words2brailles") / f"{W2B}.txt"
)
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(braille_text, encoding="utf-8")

print(braille_text)
print(f"\nSaved to: {output_path}")

your text to translate here
⠽⠕⠥⠗ ⠞⠑⠭⠞ ⠞⠕ ⠞⠗⠁⠝⠎⠇⠁⠞⠑ ⠓⠑⠗⠑

Saved to: /content/user_words2brailles/test_result4.txt


In [ ]:
%pip install pybraille

  Preparing metadata (setup.py) ... done
  Created wheel for pybraille: filename=pybraille-1.0.0-py3-none-any.whl size=3606 sha256=4cfa1a4ffc62b4b4a352115b544b0b7c9bc8e975caf69c551485059f7b6d8cb6
  Stored in directory: /root/.cache/pip/wheels/68/25/63/4a113a4d2ce339c26bd1433b9644494e10020aff1456be92ee
Successfully built pybraille


In [ ]:
import base64
import numpy as np
from PIL import Image, ImageOps
import cv2
from google.colab import output
from IPython.display import display, HTML, JSON

# ---------- Text -> Braille (self-contained) ----------
_LETTERS = dict(zip(
    "abcdefghijklmnopqrstuvwxyz",
    "⠁⠃⠉⠙⠑⠋⠛⠓⠊⠚⠅⠇⠍⠝⠕⠏⠟⠗⠎⠞⠥⠧⠺⠭⠽⠵"
))
_DIGITS = dict(zip("1234567890", "⠁⠃⠉⠙⠑⠋⠛⠓⠊⠚"))
_PUNCT = {
    " ": " ", ",": "⠂", ".": "⠲", "!": "⠖", "?": "⠦",
    "'": "⠄", "-": "⠤", ":": "⠒", ";": "⠆", "\n": "\n",
}
_CAP = "⠠"
_NUM = "⠼"
_UNKNOWN = "⬚"

def text_to_braille(text):
    out = []
    unsupported = []
    numeric_mode = False
    for ch in text:
        if ch in _DIGITS:
            if not numeric_mode:
                out.append(_NUM)
            out.append(_DIGITS[ch])
            numeric_mode = True
            continue
        numeric_mode = False
        low = ch.lower()
        if ch.isupper() and low in _LETTERS:
            out.append(_CAP + _LETTERS[low])
        elif low in _LETTERS:
            out.append(_LETTERS[low])
        elif ch in _PUNCT:
            out.append(_PUNCT[ch])
        else:
            unsupported.append(ch)
            out.append(_UNKNOWN)
    return "".join(out), sorted(set(unsupported))


# ---------- Image -> Braille (dot matrix packed into Unicode braille cells) ----------
def image_to_braille(image_bytes, dot_cols=88, dot_rows=60):
    arr = np.frombuffer(image_bytes, dtype=np.uint8)
    img = cv2.imdecode(arr, cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError("Could not read that image file.")

    pil = Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    pil = ImageOps.exif_transpose(pil).convert("L")

    # Fit into the dot grid without cropping, padding with white.
    fitted = ImageOps.contain(pil, (dot_cols, dot_rows), method=Image.Resampling.LANCZOS)
    canvas = Image.new("L", (dot_cols, dot_rows), 255)
    x = (dot_cols - fitted.width) // 2
    y = (dot_rows - fitted.height) // 2
    canvas.paste(fitted, (x, y))

    dark = np.array(canvas) < 128  # True where a dot should be raised
    h, w = dark.shape
    pad_h, pad_w = (-h) % 3, (-w) % 2
    if pad_h or pad_w:
        dark = np.pad(dark, ((0, pad_h), (0, pad_w)))
        h, w = dark.shape

    weights = [1, 8, 2, 16, 4, 32]  # dot1,dot4,dot2,dot5,dot3,dot6 (row-major 3x2 block)
    lines = []
    for by in range(0, h, 3):
        row_chars = []
        for bx in range(0, w, 2):
            block = dark[by:by + 3, bx:bx + 2].flatten()
            bits = sum(wgt for bit, wgt in zip(block, weights) if bit)
            row_chars.append(chr(0x2800 + bits))
        lines.append("".join(row_chars))
    return "\n".join(lines)


# ---------- Callbacks the front-end JS invokes ----------
def _cb_translate_text(text):
    if not text or not text.strip():
        return JSON({"ok": False, "error": "Please enter some text first."})
    braille, unsupported = text_to_braille(text)
    warning = ("Skipped unsupported characters: " + " ".join(unsupported)) if unsupported else None
    return JSON({"ok": True, "result": braille, "warning": warning})


def _cb_translate_image(data_url):
    try:
        _, b64data = data_url.split(",", 1) if "," in data_url else (None, data_url)
        raw = base64.b64decode(b64data)
        braille = image_to_braille(raw)
        return JSON({"ok": True, "result": braille, "warning": None})
    except Exception as exc:
        return JSON({"ok": False, "error": str(exc)})


output.register_callback("notebook.translate_text", _cb_translate_text)
output.register_callback("notebook.translate_image", _cb_translate_image)


# ---------- Front-end UI ----------
html_code = r"""
<style>
  #braille-app {
    font-family: 'Segoe UI', Tahoma, sans-serif;
    max-width: 980px;
    margin: 10px 0;
    padding: 22px 26px;
    border-radius: 16px;
    background: linear-gradient(160deg, #eaf7ee 0%, #d3f0dc 55%, #c3e8cf 100%);
    border: 1px solid #a9cdb2;
    box-shadow: 0 6px 18px rgba(70, 110, 80, 0.15);
    color: #274c34;
  }
  #braille-app h2 { margin: 0 0 4px 0; color: #2f5d3d; }
  #braille-app p.subtitle { margin: 0 0 16px 0; color: #4f7a5c; font-size: 13px; }
  .toggle-row { display: flex; gap: 8px; margin-bottom: 14px; }
  .toggle-btn {
    flex: 1; padding: 10px 0; border: 1.5px solid #8fbb9c;
    background: #f3fbf5; color: #3a6a4a; border-radius: 10px;
    font-weight: 600; font-size: 14px; cursor: pointer;
    transition: all 0.15s ease-in-out;
  }
  .toggle-btn:hover { background: #dff2e4; }
  .toggle-btn.active { background: #6fb98a; border-color: #5aa476; color: #ffffff; }
  #textPanel textarea {
    width: 100%; box-sizing: border-box; min-height: 90px;
    padding: 10px 12px; border-radius: 10px; border: 1.5px solid #a9cdb2;
    background: #fbfffc; color: #274c34; font-size: 14px; resize: vertical;
  }
  #imagePanel {
    padding: 18px; border: 2px dashed #8fbb9c; border-radius: 10px;
    background: #f3fbf5; text-align: center;
  }
  #imagePanel input[type="file"] { color: #3a6a4a; }
  #submitBtn {
    display: block; margin: 16px 0 6px 0; padding: 11px 22px; border: none;
    border-radius: 10px; background: #4f9c6d; color: white; font-size: 15px;
    font-weight: 600; cursor: pointer; transition: background 0.15s ease-in-out;
  }
  #submitBtn:hover { background: #3d8659; }
  #submitBtn:disabled { background: #a9cdb2; cursor: default; }
  #braille-app h3 { margin: 18px 0 8px 0; color: #2f5d3d; font-size: 15px; }
  #brailleOutput {
    display: block; width: 100%; max-width: 100%; box-sizing: border-box;
    aspect-ratio: 110 / 70; height: auto; min-height: 260px;
    font-family: 'Courier New', monospace; font-size: 13px; line-height: 1.35;
    padding: 10px; border-radius: 10px; border: 1.5px solid #7fae8f;
    background: #fbfffc; color: #1c3324; resize: vertical; overflow: auto;
    white-space: pre;
  }
  .hint { font-size: 11px; color: #6a9278; margin-top: 4px; }
</style>

<div id="braille-app">
  <h2>🌿 Braille Translator</h2>
  <p class="subtitle">Translate typed text or an uploaded image into Unicode Braille.</p>

  <div class="toggle-row">
    <button type="button" id="btnTextMode" class="toggle-btn active">📝 Text</button>
    <button type="button" id="btnImageMode" class="toggle-btn">🖼️ Image</button>
  </div>

  <div id="textPanel">
    <textarea id="textInput" placeholder="Type something to translate into Braille..."></textarea>
  </div>

  <div id="imagePanel" style="display:none;">
    <input type="file" id="imageInput" accept="image/*" />
    <div class="hint">Choose an image to convert into a Braille dot pattern.</div>
  </div>

  <button type="button" id="submitBtn">Translate ➜ ⠃⠗⠁⠊⠇⠇⠑</button>

  <h3>Braille Output</h3>
  <textarea id="brailleOutput" cols="110" rows="70" readonly></textarea>
</div>

<script>
(function () {
  const btnTextMode = document.getElementById('btnTextMode');
  const btnImageMode = document.getElementById('btnImageMode');
  const textPanel = document.getElementById('textPanel');
  const imagePanel = document.getElementById('imagePanel');
  const textInput = document.getElementById('textInput');
  const imageInput = document.getElementById('imageInput');
  const submitBtn = document.getElementById('submitBtn');
  const brailleOutput = document.getElementById('brailleOutput');

  let mode = 'text';

  function setMode(newMode) {
    mode = newMode;
    const isText = mode === 'text';
    btnTextMode.classList.toggle('active', isText);
    btnImageMode.classList.toggle('active', !isText);
    textPanel.style.display = isText ? 'block' : 'none';
    imagePanel.style.display = isText ? 'none' : 'block';
  }

  btnTextMode.addEventListener('click', function () {
    if (mode !== 'text') { setMode('text'); alert('Switched to Text mode.'); }
  });

  btnImageMode.addEventListener('click', function () {
    if (mode !== 'image') { setMode('image'); alert('Switched to Image mode.'); }
  });

  function fileToDataURL(file) {
    return new Promise(function (resolve, reject) {
      const reader = new FileReader();
      reader.onload = function () { resolve(reader.result); };
      reader.onerror = reject;
      reader.readAsDataURL(file);
    });
  }

  submitBtn.addEventListener('click', async function () {
    brailleOutput.value = '';

    if (mode === 'text') {
      const text = textInput.value;
      if (!text.trim()) { alert('Please enter some text first.'); return; }
      submitBtn.disabled = true;
      try {
        const res = await google.colab.kernel.invokeFunction('notebook.translate_text', [text], {});
        const data = res.data['application/json'];
        if (!data.ok) {
          alert('Error: ' + data.error);
        } else {
          brailleOutput.value = data.result;
          if (data.warning) alert(data.warning);
          alert('Text translated to Braille!');
        }
      } catch (err) {
        alert('Something went wrong: ' + err);
      } finally {
        submitBtn.disabled = false;
      }
    } else {
      if (!imageInput.files.length) { alert('Please choose an image first.'); return; }
      submitBtn.disabled = true;
      try {
        const dataUrl = await fileToDataURL(imageInput.files[0]);
        const res = await google.colab.kernel.invokeFunction('notebook.translate_image', [dataUrl], {});
        const data = res.data['application/json'];
        if (!data.ok) {
          alert('Error: ' + data.error);
        } else {
          brailleOutput.value = data.result;
          alert('Image translated to Braille!');
        }
      } catch (err) {
        alert('Something went wrong: ' + err);
      } finally {
        submitBtn.disabled = false;
      }
    }
  });
})();
</script>
"""

display(HTML(html_code))